### Taxonomy Barplot


Import libraries and set working directory

In [8]:
import os
import pandas as pd
import qiime2 as q2
from qiime2 import Visualization
%matplotlib inline
import seaborn as sns
import matplotlib.pyplot as plt
from biom import Table

%cd /home/jovyan/MicroOdyssey_clean

meta_dir= "data"
bracken_dir= "output/02_taxonomicclassification"
output_dir= "output/02_taxonomicclassification"

/home/jovyan/MicroOdyssey_clean


In [9]:
import os

print(os.getcwd())
print(os.path.abspath(meta_dir))
print(os.path.exists(f"{meta_dir}/metadata.tsv"))

/home/jovyan/MicroOdyssey_clean
/home/jovyan/MicroOdyssey_clean/data
True


Build taxa barplot

In [10]:
! qiime taxa barplot \
  --i-table $bracken_dir/bracken-table.qza \
  --i-taxonomy $bracken_dir/bracken-taxonomy.qza \
  --m-metadata-file $meta_dir/metadata.tsv \
  --o-visualization $output_dir/figures/bracken_barplot_ageordered.qzv

Saved Visualization to: output/02_taxonomicclassification/figures/bracken_barplot_ageordered.qzv


In [ ]:
Visualization.load(f"{output_dir}/figures/bracken_barplot_ageordered.qzv")

### Genus Heatmap

In [14]:
import pandas as pd

meta = pd.read_csv("data/metadata.tsv", sep="\t")
#rename ages for correct order in taxa barplot
meta["age_group_ordered"] = meta["age_months"].map({
    0: "00_Birth",
    2: "02_months",
    6: "06_months",
    12: "12_months",
    24: "24_months"
})

meta.to_csv("data/metadata_heatmap.tsv", sep="\t", index=False)

In [15]:
! qiime taxa collapse \
  --i-table $bracken_dir/bracken-table.qza \
  --i-taxonomy $bracken_dir/bracken-taxonomy.qza \
  --p-level 7 \
  --o-collapsed-table $output_dir/genus_table.qza

Saved FeatureTable[Frequency] to: output/02_taxonomicclassification/genus_table.qza


In [16]:
#Only inlcude genus with 0.01 abundance and 10% prevalence. 
! qiime feature-table filter-features-conditionally \
  --i-table $output_dir/genus_table.qza \
  --p-abundance 0.01 \
  --p-prevalence 0.10 \
  --o-filtered-table $output_dir/genus_filtered.qza

Saved FeatureTable[Frequency] to: output/02_taxonomicclassification/genus_filtered.qza


In [ ]:
#generate heatmap with the filtered genus file
! qiime feature-table heatmap \
  --i-table $output_dir/genus_filtered.qza \
  --m-sample-metadata-file $meta_dir/metadata_heatmap.tsv \
  --m-sample-metadata-column age_group_ordered \
  --p-cluster features \
  --o-visualization $output_dir/genus_heatmap.qzv

In [ ]:
Visualization.load(f"{results_dir}/genus_heatmap.qzv")

In [ ]:
#Select the most abundant named genera
top_genera = [
    name for name in age_means.mean().sort_values(ascending=False).index
    if name.split(";")[-1].replace("g__", "").strip() not in ["", "—", "__"]
][:25]

heatmap_data = age_means[top_genera].copy()

#Shorten genus names
heatmap_data.columns = [
    name.split(";")[-1].replace("g__", "")
    for name in heatmap_data.columns
]

# Transpose the table: genera become rows, ages become columns
heatmap_data = heatmap_data.T

#Apply log transformation to make smaller abundances visible
import numpy as np

heatmap_log = np.log10(heatmap_data + 0.1)

plt.figure(figsize=(9, 7))

sns.heatmap(
    heatmap_log,
    cmap="mako",
    annot=heatmap_data,
    fmt=".1f",
    linewidths=0.5,
    cbar_kws={"label": "log10(mean relative abundance (%) + 0.1)"}
)

plt.xlabel("Infant age")
plt.ylabel("Genus")
plt.title("Genus-level microbiome development")

plt.xticks(
    ticks=np.arange(5) + 0.5,
    labels=["Birth", "2 m", "6 m", "12 m", "24 m"],
    rotation=0
)

plt.yticks(rotation=0)
plt.tight_layout()
plt.savefig("Top_genera_heatmap.png", dpi=300, bbox_inches="tight")
plt.show()

### Run Ancom-BC2 for statistical evidence

Acombc - 2 RUN 

In [16]:
! qiime composition ancombc2 \
    --i-table $data_dir/genus_table.qza \
    --m-metadata-file $data_dir/metadata_heatmap.tsv \
    --p-fixed-effects-formula "age_group_ordered" \
    --p-random-effects-formula "(1 | host_id)" \
    --p-reference-levels age_group_ordered::00_Birth \
    --p-p-adjust-method BH \
    --p-prevalence-cutoff 0.1 \
    --o-ancombc2-output $data_dir/ancombc2_age.qza

Saved FeatureData[ANCOMBC2Output] to: data/ancombc2_age.qza


In [18]:
! qiime composition da-barplot \
    --i-data $data_dir/ancombc2_age.qza \
    --o-visualization $data_dir/ancombc2_age_barplot.qzv

Saved Visualization to: data/ancombc2_age_barplot.qzv


In [ ]:
Visualization.load(f"{results_dir}/ancombc2_age_barplot.qzv")

In [20]:
! qiime composition tabulate \
    --i-data $data_dir/ancombc2_age.qza \
    --o-visualization $data_dir/ancombc2_age_results.qzv

Saved Visualization to: data/ancombc2_age_results.qzv


In [1]:
#Visualization.load(f"{results_dir}/ancombc2_age_results.qzv")

In [ ]:
! qiime composition ancombc2 \
    --i-table $data_dir/genus_table.qza \
    --m-metadata-file $data_dir/metadata_heatmap.tsv \
    --p-fixed-effects-formula "age_group_ordered" \
    --p-random-effects-formula "(1 | host_id)" \
    --p-reference-levels age_group_ordered::00_Birth \
    --p-p-adjust-method BH \
    --p-prevalence-cutoff 0.1 \
    --o-ancombc2-output $data_dir/ancombc2_age.qza